# broadcasting rules — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five broadcasting patterns that ramp from predicting the result shape → row-vector broadcast → column-vector broadcast → targeted axis insertion → outer product via broadcast. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**Three patterns you reach for constantly:**
- **Row broadcast** — `(N, D) + (D,)` works automatically. Adds a per-feature bias.
- **Column broadcast** — `(N, D) * w` where `w` is `(N,)` fails. Reshape `w` to `(N, 1)` first.
- **Axis insertion** — `unsqueeze` / `[:, None]` / `reshape` are all valid ways to insert a size-1 axis where broadcasting needs it.

### Exercise 1 — predict the broadcast shape

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall and apply the right-align broadcasting rule on two shape tuples.
> Keywords: shape-rule, right-align, incompatibility
> ```

**KCs targeted:** `predict-broadcast-shape`

Implement `ex1_broadcast_shape(shape_a, shape_b)` to return the shape that would result from broadcasting two tensors of the given shapes, OR raise `ValueError` if they're incompatible.

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes, left-pad the shorter with 1s.
2. For each pair `(a, b)` of aligned axes: keep if equal; if exactly one is 1, use the other; otherwise → incompatible.

Return the result as a tuple of ints.

**Examples:**
- `(3, 4)` and `(4,)` → `(3, 4)`
- `(2, 1, 3)` and `(5, 3)` → `(2, 5, 3)`
- `(3, 4)` and `(3,)` → `ValueError` (right-align mismatch on last axis)

In [ ]:
def ex1_broadcast_shape(shape_a, shape_b):
    """Return broadcasted shape (tuple) or raise ValueError if incompatible."""
    raise NotImplementedError()


def _test_ex1():
    # Compatible cases
    assert ex1_broadcast_shape((3, 4), (4,)) == (3, 4)
    assert ex1_broadcast_shape((3, 4), (1, 4)) == (3, 4)
    assert ex1_broadcast_shape((3, 1), (4,)) == (3, 4)
    assert ex1_broadcast_shape((2, 1, 3), (5, 3)) == (2, 5, 3)
    assert ex1_broadcast_shape((1,), (5, 6, 7)) == (5, 6, 7)
    assert ex1_broadcast_shape((), (5,)) == (5,)

    # Incompatible cases — must raise ValueError
    raised = False
    try:
        ex1_broadcast_shape((3, 4), (3,))   # right-align: (3,4) vs (1,3) → mismatch at axis -1
    except ValueError:
        raised = True
    assert raised, 'should have raised ValueError for incompatible shapes (3,4) vs (3,)'

    raised = False
    try:
        ex1_broadcast_shape((2, 3), (4, 3))
    except ValueError:
        raised = True
    assert raised, 'should have raised ValueError for (2,3) vs (4,3)'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_shape(shape_a, shape_b):
    a = list(shape_a)
    b = list(shape_b)
    n = max(len(a), len(b))
    a = [1] * (n - len(a)) + a
    b = [1] * (n - len(b)) + b
    out = []
    for ai, bi in zip(a, b):
        if ai == bi:
            out.append(ai)
        elif ai == 1:
            out.append(bi)
        elif bi == 1:
            out.append(ai)
        else:
            raise ValueError(f'incompatible axes: {ai} vs {bi}')
    return tuple(out)
```

**Why right-align?** Trailing axes correspond to the fastest-varying memory layout. Aligning shapes from the right means a `(D,)` vector broadcasts across rows of an `(N, D)` matrix, which is the natural 'one weight per feature' case. Left-align would have given you 'one weight per sample' instead — a different, much less common need.
</details>

### Exercise 2 — row-vector broadcast (add bias to a batch)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply 1-D broadcast across the leading axis of a 2-D batch (per-feature bias).
> Keywords: row-broadcast, bias, per-feature
> ```

**KCs targeted:** `broadcast-row-vector`

Implement `ex2_add_bias(x, b)` to add a per-feature bias to a batch.

Input shapes: `x` is `(N, D)`, `b` is `(D,)`. Output shape: `(N, D)`. Every row of the output should equal `x[n] + b`.

This is just `x + b` — but the point is to recognize that right-align broadcasting handles it for you. No `unsqueeze`, no manual broadcast — the rule does the work.

In [ ]:
def ex2_add_bias(x: Tensor, b: Tensor) -> Tensor:
    """Add per-feature bias. (N, D) + (D,) → (N, D)."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(2 * 3).reshape(2, 3).float()       # (2, 3)
    b = t.tensor([10.0, 20.0, 30.0])                # (3,)
    y = ex2_add_bias(x, b)
    assert y.shape == (2, 3), f'expected (2,3), got {tuple(y.shape)}'
    expected = t.tensor([[10.0, 21.0, 32.0], [13.0, 24.0, 35.0]])
    assert t.equal(y, expected), f'value mismatch: {y.tolist()}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_add_bias(x: Tensor, b: Tensor) -> Tensor:
    return x + b
```
</details>

### Exercise 3 — column-vector broadcast (per-sample scale)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `unsqueeze(1)` / `[:, None]` to make a 1-D per-sample weight broadcast across the feature axis of a 2-D batch.
> Keywords: column-broadcast, per-sample, unsqueeze
> ```

**KCs targeted:** `broadcast-column-vector`, `broadcast-via-unsqueeze`

Implement `ex3_scale_rows(x, w)` to multiply each row of a batch by a per-sample scalar.

Input shapes: `x` is `(N, D)`, `w` is `(N,)`. Output shape: `(N, D)`. Every row of the output should equal `x[n] * w[n]`.

**Watch out:** `x * w` does NOT work — right-align broadcasting tries to match `(N,)` against `(N, D)`'s last axis `D`. You need to reshape `w` to `(N, 1)` first via `w.unsqueeze(1)` or `w[:, None]`.

In [ ]:
def ex3_scale_rows(x: Tensor, w: Tensor) -> Tensor:
    """Per-sample scale. (N, D) * (N,) → (N, D) via unsqueeze."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(2 * 3).reshape(2, 3).float()        # (2, 3)
    w = t.tensor([2.0, 10.0])                        # (2,)
    y = ex3_scale_rows(x, w)
    assert y.shape == (2, 3), f'expected (2,3), got {tuple(y.shape)}'
    expected = t.tensor([[0.0, 2.0, 4.0], [30.0, 40.0, 50.0]])
    assert t.equal(y, expected), f'value mismatch: {y.tolist()}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_scale_rows(x: Tensor, w: Tensor) -> Tensor:
    return x * w.unsqueeze(1)
```

**Why does `x * w` fail with N≠D?** Right-align takes `(N,)` and tries to broadcast it against `(N, D)`'s last axis (`D`). If `N == D` it accidentally works (and gives the WRONG result — it scales by feature instead of by sample). With `N ≠ D` you get a shape error. `unsqueeze(1)` makes `w` into `(N, 1)` so it right-aligns as `(N, 1)` vs `(N, D)` → `(N, D)`. Always be explicit about which axis you're broadcasting along.
</details>

### Exercise 4 — insert a missing axis where broadcast fails

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply targeted axis insertion (`unsqueeze`) to make a 3-D + 1-D broadcast work along the desired axis.
> Keywords: unsqueeze, axis-insertion, shape-debug
> ```

**KCs targeted:** `broadcast-via-unsqueeze`

Implement `ex4_scale_channel(x, w)` to scale each feature channel of a feature-map batch by a per-channel weight.

Input shapes: `x` is `(B, C, H, W)`, `w` is `(C,)`. Output shape: `(B, C, H, W)`. Each `out[b, c, :, :] == x[b, c, :, :] * w[c]`.

Right-align would try to broadcast `(C,)` against the trailing `W` axis — wrong. Reshape `w` to insert size-1 axes where they need to be so the broadcast targets the channel axis instead.

**Hint:** the right shape for `w` to broadcast against `(B, C, H, W)` along the channel axis is `(1, C, 1, 1)`. Use `w.reshape(...)` or chained `unsqueeze` calls — both are fine.

In [ ]:
def ex4_scale_channel(x: Tensor, w: Tensor) -> Tensor:
    """Per-channel scale. (B, C, H, W) * (C,) → (B, C, H, W) via axis insertion."""
    raise NotImplementedError()


def _test_ex4():
    x = t.ones(2, 3, 4, 5)
    w = t.tensor([1.0, 2.0, 3.0])  # (C=3,)
    y = ex4_scale_channel(x, w)
    assert y.shape == (2, 3, 4, 5), f'expected (2,3,4,5), got {tuple(y.shape)}'
    # Channel 0 should be 1.0 everywhere, channel 1 should be 2.0, channel 2 should be 3.0.
    assert t.allclose(y[:, 0, :, :], t.ones(2, 4, 5)), 'channel 0 not scaled by w[0]=1'
    assert t.allclose(y[:, 1, :, :], 2.0 * t.ones(2, 4, 5)), 'channel 1 not scaled by w[1]=2'
    assert t.allclose(y[:, 2, :, :], 3.0 * t.ones(2, 4, 5)), 'channel 2 not scaled by w[2]=3'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_scale_channel(x: Tensor, w: Tensor) -> Tensor:
    return x * w.reshape(1, -1, 1, 1)
```

**Three equivalent forms** for inserting axes:
- `w.reshape(1, -1, 1, 1)` — explicit; `-1` infers C.
- `w[None, :, None, None]` — slice-syntax `None` is shorthand for `unsqueeze`.
- `w.unsqueeze(0).unsqueeze(-1).unsqueeze(-1)` — chained, error-prone in higher dims.

Pick whichever reads clearest at the call site. `reshape` is usually the most explicit for >2 axis insertions.
</details>

### Exercise 5 — outer product via column×row broadcast

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize column-vector broadcast + row-vector broadcast to compute the outer product of two 1-D tensors without `torch.outer`.
> Keywords: outer-product, column-times-row, integration, multi-kc
> ```

**KCs targeted:** `broadcast-row-vector`, `broadcast-column-vector`, `broadcast-outer-product`

Implement `ex5_outer(u, v)` to compute the outer product of two 1-D tensors.

Input shapes: `u` is `(N,)`, `v` is `(M,)`. Output shape: `(N, M)`. Each `out[i, j] == u[i] * v[j]`.

**Use broadcasting only** — no `torch.outer`, no `einsum`, no `unsqueeze` + matmul. Strategy: reshape `u` to a column `(N, 1)` and `v` to a row `(1, M)`, then multiply. The right-align rule produces `(N, M)`.

Equivalent to `torch.outer(u, v)`.

> ⚠️ **Integrative exercise.** This combines 3+ KCs (column-broadcast, row-broadcast, axis insertion) in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_outer(u: Tensor, v: Tensor) -> Tensor:
    """Outer product via broadcasting. (N,) * (M,) → (N, M)."""
    raise NotImplementedError()


def _test_ex5():
    u = t.tensor([1.0, 2.0, 3.0])     # (3,)
    v = t.tensor([10.0, 20.0])         # (2,)
    M = ex5_outer(u, v)
    assert M.shape == (3, 2), f'expected (3,2), got {tuple(M.shape)}'
    expected = t.tensor([[10.0, 20.0], [20.0, 40.0], [30.0, 60.0]])
    assert t.equal(M, expected), f'value mismatch: {M.tolist()}'
    # Also compare to torch.outer ground truth.
    assert t.equal(M, t.outer(u, v)), 'differs from t.outer(u, v)'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_outer(u: Tensor, v: Tensor) -> Tensor:
    return u.unsqueeze(1) * v.unsqueeze(0)
```

**Reading the pattern.**
- `u.unsqueeze(1)` → shape `(N, 1)` (column vector).
- `v.unsqueeze(0)` → shape `(1, M)` (row vector).
- `(N, 1) * (1, M)` right-aligns as `(N, 1) * (1, M)` → broadcasts to `(N, M)`.

Each broadcast tile fills a row (from `v`) or a column (from `u`); their elementwise product gives `u[i] * v[j]` at every position.

**Equivalent forms:** `u[:, None] * v[None, :]` (same thing, slice syntax); `torch.einsum('i,j->ij', u, v)`; `torch.outer(u, v)`. The broadcasting form is worth knowing because it generalizes to higher ranks (e.g. batched outer product) without changing the mental model.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()